# Ask the IoT agents: features, drift and RAG

This notebook sends real questions to the private kagent deployment. The retrieval agent calls three MCP tools: current Feast-backed features, the model prediction, and versioned reference passages. The drift agent calls the cohort-comparison MCP tool. The model decides and invokes these calls inside its native agent runtime; the notebook does not supply a prewritten answer.

Run it through `scripts/run-agent-notebook.py` after genuine feature preparation. The runner records the source seal, native service identity, data lease and Jupyter execution. An expired cache should fail rather than silently appear fresh. All machines, measurements and reference documents in this lab are simulated.

## 1. Connect to the actual lab

The context file contains paths and a private controller address, not MCP credentials. The internal A2A interface is reachable only inside the lab; authenticated product ingress is a separate requirement. Each request gets a new task/context identity, and original request/response bytes are retained.

In [ ]:
from pathlib import Path
import json
import os
import sys

context = json.loads(Path(os.environ['IOT_AGENT_NOTEBOOK_CONTEXT']).read_bytes())
source = Path(context['source_root']).resolve()
output = Path(context['output_root']).resolve()
sys.path.insert(0, str(source))
from agents.probe import QUESTIONS, run
from agents.answer_acceptance import final_answer, native_tools, verify_specialist

base_url = context['controller_base_url']
lease = context['feature_lease']
print(json.dumps({'source_files': len(context['source_files']),
                  'model_version': lease['model_version'],
                  'prepared_wall_utc': lease['prepared_wall_utc'],
                  'lease_expires_utc': lease['lease_expires_utc'],
                  'simulated': True}, indent=2))

## 2. Ask the retrieval specialist

The prompt asks about one pump and requests evidence. A completed HTTP/A2A task alone does not prove a correct answer. We therefore check the final JSON against the actual paired MCP calls and outputs.

In [ ]:
print(QUESTIONS['retrieval'])
assert run(base_url, output / 'retrieval', ['retrieval'], specialist_timeout=300) == 0
retrieval_reply = json.loads((output / 'retrieval/retrieval-response.json').read_bytes())
retrieval_check = verify_specialist(retrieval_reply, 'retrieval', knowledge_root=source / 'knowledge')
retrieval_answer = final_answer(retrieval_reply['result'])
print(json.dumps(retrieval_answer, indent=2))
print(json.dumps(retrieval_check, indent=2))

## 3. Follow the feature and prediction provenance

`get_machine_features` reaches the existing Feature API, which reads the genuinely materialized Feast online store. `predict_failure` returns the feature trace used by model version 6. The checker requires that both refer to the same machine snapshot. Event time is simulated; preparation time is real wall time. The score is not a calibrated real-world probability.

In [ ]:
retrieval_tools, retrieval_counts = native_tools(retrieval_reply['result'], 'retrieval')
feature = retrieval_tools['get_machine_features']['output']
prediction = retrieval_tools['predict_failure']['output']
assert prediction['feature_trace'] == feature
print(json.dumps({'native_tool_names': sorted(retrieval_tools),
                  'feature': feature,
                  'model': prediction['model'],
                  'prediction': prediction['prediction'],
                  'actual_event_counts': retrieval_counts}, indent=2))

## 4. Inspect the RAG evidence

`search_knowledge` embeds the question and retrieves versioned passages from the retained Feast/Milvus knowledge snapshot. The answer checker reads the authored source files, hashes them and checks the cited text. This proves the shown answer cites returned source material; the small example does not establish broad retrieval quality or real equipment safety.

In [ ]:
knowledge = retrieval_tools['search_knowledge']['output']
matches = knowledge['matches']
print(json.dumps({'query': retrieval_tools['search_knowledge']['arguments'],
                  'snapshot_sha256': knowledge['snapshot_sha256'],
                  'embedding_revision': knowledge['embedding_revision'],
                  'source_type': knowledge['source_type'],
                  'returned_passages': matches,
                  'answer_document_ids': retrieval_answer['document_ids']}, indent=2))
assert set(retrieval_answer['document_ids']) == {row['document_id'] for row in matches}
assert retrieval_answer['vibration_alone_proves_failure'] is False

## 5. Ask the drift specialist

This question names all 32 unique machines and one stable window. The MCP service retrieves their current features and compares the cohort with a pinned reference. Replaying a window must not increase the number of machines. Input drift alone does not prove a loss of model accuracy or a pump failure.

In [ ]:
print(QUESTIONS['drift'])
assert run(base_url, output / 'drift', ['drift'], specialist_timeout=300) == 0
drift_reply = json.loads((output / 'drift/drift-response.json').read_bytes())
drift_check = verify_specialist(drift_reply, 'drift', knowledge_root=source / 'knowledge')
drift_answer = final_answer(drift_reply['result'])
print(json.dumps(drift_answer, indent=2))
print(json.dumps(drift_check, indent=2))

In [ ]:
drift_tools, drift_counts = native_tools(drift_reply['result'], 'drift')
native_drift = drift_tools['assess_feature_drift']
print(json.dumps({'native_arguments': native_drift['arguments'],
                  'native_assessment': native_drift['output'],
                  'actual_event_counts': drift_counts}, indent=2))
assert len(native_drift['arguments']['machine_ids']) == len(set(native_drift['arguments']['machine_ids'])) == 32
assert drift_answer['unique_rows'] == 32
assert drift_answer['drift_establishes_accuracy_loss'] is False

## 6. State exactly what passed

We have executed two actual specialist requests and checked their final answers against native tool histories. One request demonstrates both Feast-backed machine retrieval and RAG; the other demonstrates drift. Coordinator delegation, elasticity, network boundaries, model benchmarks and API quality have separate experiments. This notebook does not replace them or complete the full coursework.

In [ ]:
checks = {'passed': retrieval_check['passed'] and drift_check['passed'],
          'retrieval': retrieval_check, 'drift': drift_check,
          'native_successful_tool_events': retrieval_counts['successful_tool_events'] + drift_counts['successful_tool_events'],
          'feature_and_prediction_same_snapshot': prediction['feature_trace'] == feature,
          'rag_source_citations_verified': True,
          'unique_drift_machines': drift_answer['unique_rows'],
          'coursework_complete': False}
(output / 'answer-checks.json').write_text(json.dumps(checks, indent=2) + '\n', encoding='utf-8')
print(json.dumps(checks, indent=2))
assert checks['passed']